# Sentence Window Retrieval for Advanced RAG

This notebook explains and demonstrates the Sentence Window Retrieval technique using LlamaIndex, and compares it against a standard (base) chunking approach so the difference in retrieval quality is easy to see.

## What problem does this solve?

In a typical Retrieval Augmented Generation (RAG) pipeline, a document is split into chunks, each chunk is embedded into a vector, and at query time the chunks whose embeddings are closest to the question are retrieved and passed to the language model.

There is a tension in choosing the chunk size.

1. Small chunks (for example, a single sentence) produce very focused embeddings. A single sentence usually expresses one idea clearly, so the embedding is precise and matches the query well.
2. Small chunks also throw away surrounding context. A single sentence read in isolation can be ambiguous or missing details that appear in the sentences right before or after it.
3. Large chunks preserve context but dilute the embedding, since many unrelated ideas get mixed into one vector, which can make retrieval less accurate.

Sentence Window Retrieval resolves this tension with a simple idea. Embed and match on a single sentence (so the search itself stays precise), but when a sentence is retrieved, replace it with a window of the surrounding sentences (so the language model actually sees the full context needed to answer well). This notebook builds both a sentence window pipeline and a normal chunking pipeline side by side, so their answers to the same question can be compared directly.

## What this notebook covers

1. Installing and importing the required libraries.
2. Configuring the embedding model and the language model.
3. Loading the source document.
4. Building two different node parsing strategies, the Sentence Window parser and a standard Sentence Splitter.
5. Inspecting how each strategy structures the underlying nodes.
6. Indexing both sets of nodes into a Qdrant vector store.
7. Building query engines for each strategy, including the special postprocessor that restores full context for the sentence window approach.
8. Asking the same question against both engines and comparing the answers.
9. A conclusion that ties the observations together.


## Section 1. Environment setup

Before any retrieval logic can run, the required packages need to be installed. The cells below install LlamaIndex itself along with three optional integration packages that this notebook depends on.

1. `llama_index` is the core framework that provides document loaders, node parsers, indexes, and query engines.
2. The HuggingFace embeddings integration package lets LlamaIndex use a HuggingFace sentence transformer model to turn text into embedding vectors.
3. The Qdrant vector store integration package lets LlamaIndex store and search those embedding vectors inside a Qdrant vector database, instead of only keeping them in memory.
4. The Gemini integration package lets LlamaIndex send prompts to Google's Gemini language model and use its responses to answer questions.

The exact install commands for each package are shown in the code cells right below this explanation.

Each package is installed separately so it is clear exactly what each one contributes to the pipeline.


In [ ]:
!pip install llama_index

In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
!pip install llama-index-vector-stores-qdrant

In [ ]:
!pip install llama-index-llms-gemini

In [ ]:
!pip install -q llama-index google-generativeai

### A note on autoreload

The two lines below are a Jupyter/Colab convenience feature, not part of the RAG pipeline itself. `autoreload` automatically reloads any Python modules that change on disk, which is useful while iterating on helper code in separate files. It has no effect on the retrieval logic in this notebook, but it is kept here since it is part of a typical development workflow.


In [ ]:
%load_ext autoreload
%autoreload 2

## Section 2. Imports

This section brings in every class that the pipeline needs. Each import is grouped with a short explanation of what the class does and why it matters for this notebook, rather than being dumped in a single block, so the purpose of each dependency is clear before it gets used later on.


### Standard library imports

`os` is used to set environment variables such as API keys, `sys` and `pprint` are general purpose utilities used for debugging output.


In [ ]:
import os
import sys
import pprint

### Core LlamaIndex classes

1. `VectorStoreIndex` builds a searchable vector index out of a set of nodes or documents.
2. `SimpleDirectoryReader` loads plain text (and other common formats) from a folder or a list of files into LlamaIndex `Document` objects.
3. `load_index_from_storage` and `StorageContext` are used to persist an index to disk and reload it later, so it does not need to be rebuilt from scratch every session.
4. `ServiceContext` bundles together the language model, the embedding model, and the node parser that an index should use. It ensures that whichever parser or model was configured is consistently applied every time the index processes documents or queries.
5. `Document` is the basic unit LlamaIndex uses to represent a piece of source text before it is split into nodes.


In [ ]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    load_index_from_storage,
    StorageContext,
    ServiceContext,
    Document)

### SentenceWindowNodeParser

This is the class that implements the sentence window technique described in the overview. It splits a document into individual sentences, and for every sentence it stores a `window` of the surrounding sentences as metadata.

For example, with a window size of 3, the parser produces overlapping groups such as:

Sentence 1, Sentence 2, Sentence 3
Sentence 2, Sentence 3, Sentence 4
Sentence 3, Sentence 4, Sentence 5

Only the single central sentence is embedded and used for similarity search. The full window is kept alongside it so it can be swapped in later, after retrieval, to give the language model more context than the single sentence alone would provide.


In [ ]:
from llama_index.core.node_parser import SentenceWindowNodeParser

### SentenceSplitter

`SentenceSplitter` is the standard, or base, way of chunking text in LlamaIndex. It groups multiple consecutive sentences into a single chunk up to a target size, and both embeds and retrieves that whole chunk as one unit. This notebook uses it as the baseline that the sentence window approach is compared against, since it represents a typical RAG chunking approach without any special context expansion step.


In [ ]:
from llama_index.core.text_splitter import SentenceSplitter

### HuggingFaceEmbedding

This wraps a HuggingFace sentence transformer model so LlamaIndex can call it to turn any piece of text into a numeric embedding vector. Both the sentence window pipeline and the base pipeline use the same embedding model, so any difference in the final answers can be attributed to the chunking and context strategy rather than to a difference in embeddings.


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

### MetadataMode

`MetadataMode` is an enum that controls which metadata fields are included when LlamaIndex turns a node into text, for example when building the string that is sent to the embedding model or to the language model. It matters here because the sentence window technique stores the window of surrounding sentences as metadata, and the code needs a way to say whether that metadata should be included, excluded, or handled in some other specific way at each stage of the pipeline.


In [ ]:
from llama_index.core.schema import MetadataMode

### MetadataReplacementPostProcessor

This is the class that performs the second half of the sentence window technique. After the vector search finds the single best matching sentence, this postprocessor looks at that node's metadata and replaces the node's text with the full window of surrounding sentences stored under the `window` metadata key. Without this step, the sentence window index would only ever return single, possibly under informative sentences to the language model, which would defeat the purpose of storing the wider window in the first place.


In [ ]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

## Section 3. Configuring the embedding model and the language model

Two different models are required for this pipeline.

1. An embedding model, which turns text into vectors so that semantic similarity can be measured. This notebook uses a HuggingFace sentence transformer, `sentence transformers/all mpnet base v2`, with a maximum input length of 512 tokens.
2. A language model, which reads the retrieved context and actually writes the final answer in natural language. This notebook uses Google's Gemini Pro model through the `llama index llms gemini` integration.

Both models are configured once and then reused by every index and query engine built later in the notebook, so the comparison between the sentence window approach and the base approach only varies the chunking and context strategy, not the underlying models.


In [ ]:
# Loading embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-mpnet-base-v2", max_length=512)

### Setting the Gemini API key

A Google API key is required to call the Gemini model. The key should never be hard coded directly into a notebook, since anyone who sees the notebook file (for example after pushing it to GitHub) would then have access to that key. Instead, store the key as an environment variable, either by setting it outside the notebook, by using a Colab secret, or by loading it from a local `.env` file with a library such as `python dotenv`.

The placeholder below should be replaced with a real key at runtime, and the actual value should never be committed to source control.


In [ ]:
import os

# Replace the placeholder below with your own key, or better yet, load it from an
# environment variable or a secrets manager instead of typing it directly into the notebook.
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

### Listing available Gemini models

This is a quick sanity check. It lists every Gemini model available to the configured API key that supports the `generateContent` method, which is the method used for standard text generation. Running this confirms the API key is valid and shows exactly which model name to use next.


In [ ]:
import google.generativeai as genai
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

### Creating the LLM object and testing it

Here the `Gemini` wrapper is instantiated with a specific model name, and a single test message is sent through the chat interface. This step confirms the language model itself is reachable and responding before it gets wired into the full retrieval pipeline.


In [ ]:
from llama_index.llms.gemini import Gemini
llm = Gemini(model="models/gemini-pro")

In [ ]:
from llama_index.core.llms import ChatMessage
messages = [
    ChatMessage(role="user", content="Hello friend!"),
]

response = llm.chat(messages)
print(response)

## Section 4. Ingesting the source data

With both models configured, the next step is to load the raw text that will be indexed. This notebook uses a Game of Thrones book excerpt as the source document, which works well for this demonstration since it contains rich descriptive passages where the surrounding context genuinely changes how a sentence should be interpreted.

`SimpleDirectoryReader` reads the file (or files) at the given path and returns a list of `Document` objects, one per file, each one containing the raw text plus some basic metadata such as the file name.


In [ ]:
documents = SimpleDirectoryReader(input_files=["/content/my_data/got_book.txt"]).load_data()

### Inspecting what was loaded

Before doing anything else with the documents, it is worth confirming how many were actually loaded and what metadata is attached to them. This is a simple but important sanity check, since a mistake in the file path or the reader configuration would silently produce an empty or incorrect document list, and any downstream retrieval results would be meaningless.


In [ ]:
documents

In [ ]:
# Inspect the documents
print("length of doc: " + str(len(documents)))
print("----")

In [ ]:
documents[0].metadata

## Section 5. Splitting the document into nodes

A node parser is responsible for breaking a document down into smaller, indexable units called nodes. Think of a full document as an entire book. A node parser plays the role of a chapter divider, or even a sentence level divider, depending on how granular the split needs to be.

The key job of a node parser is to decide the size and the boundaries of each chunk, and to decide what metadata (if any) travels along with each chunk. This notebook builds two node parsers side by side so their outputs can be compared directly.

1. `SentenceWindowNodeParser`, configured with a window size of 3 sentences on either side of the central sentence. Each resulting node stores the single sentence as its text for embedding purposes, and stores the surrounding window (labeled `window`) and the original single sentence text (labeled `original_text`) as metadata.
2. `SentenceSplitter`, used here with its default settings, which groups several consecutive sentences into a single, larger chunk with no special metadata for later expansion.


### Building the sentence window parser

The parameters below configure exactly how the window is built.

1. `window_size=3` means three sentences before and three sentences after the central sentence are included in the window.
2. `window_metadata_key="window"` sets the metadata field name that will hold the full window of text.
3. `original_text_metadata_key="original_text"` sets the metadata field name that will hold just the single original sentence, separate from the window.


In [ ]:
# create the sentence window node parser w/ default settings
sentence_node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)

In [ ]:
nodes = sentence_node_parser.get_nodes_from_documents(documents)

### Building the base sentence splitter

For comparison, a plain `SentenceSplitter` is created with its default chunk size and no window metadata. This represents the more common, simpler chunking approach used in many RAG pipelines.


In [ ]:
base_node_parser = SentenceSplitter()

In [ ]:
base_nodes = base_node_parser.get_nodes_from_documents(documents)

### Comparing the number of nodes produced

Since the sentence window parser creates one node per sentence, while the base splitter groups many sentences into each chunk, the sentence window approach is expected to produce noticeably more nodes from the same source document. Counting both confirms this difference in granularity.


In [ ]:
len(nodes)

In [ ]:
len(base_nodes)

### Looking at the same node index from both parsers

Printing node number 7 from each list side by side makes the structural difference between the two parsing strategies concrete. The sentence window node should look like a single, short sentence, while the base node should look like a longer, multi sentence chunk.


In [ ]:
print("---------")
print("SENTENCE NODES")
print("---------")
print(nodes[7])
print("---------")
print("BASE NODES")
print("---------")
print(base_nodes[7])

In [ ]:
nodes[7].text

In [ ]:
base_nodes[7].text

### Inspecting the window metadata directly

Here the `window` metadata field for node 7 is shown on its own. This is the block of surrounding sentences that will later be substituted back in after retrieval, and it clearly contains more surrounding dialogue and description than the single embedded sentence by itself.


In [ ]:
"""
window: 'Gared did not rise to the bait.  He was an old man, past fifty, and he had seen the lordlings come and go. \r\n "Dead is dead," he said.  "We have no business with the dead." \r\n "Are they dead?"  Royce asked softly.  "What proof have we?" \r\n',
"""


The `original_text` field, by contrast, stores only the single sentence that was actually embedded and matched against the query, which in this case is just:

`'original_text': '"We have no business with the dead." \r\n'`

Comparing these two fields side by side is the clearest illustration of the core idea behind sentence window retrieval. The narrow `original_text` sentence is what gets searched, while the wider `window` is what eventually gets shown to the language model.


### Viewing the complete node objects

Converting a node to a dictionary shows every attribute LlamaIndex stores for it, including its id, its relationships to neighboring nodes, and all of its metadata fields. This is useful for understanding the full internal structure LlamaIndex is working with, beyond just the `text` and `window` fields inspected above.


In [ ]:
dict(nodes[7])

In [ ]:
dict(base_nodes[7])

## Section 6. Binding models and parsers together with a service context

A `ServiceContext` bundles the language model, the embedding model, and the node parser into a single configuration object. Passing a service context into an index build call ensures that index consistently uses that exact combination of models and parsing strategy, both when it is first created and whenever it is queried afterward.

Two separate service contexts are created here, one for each parsing strategy, so the sentence window pipeline and the base pipeline remain fully independent of one another while still sharing the same underlying language model and embedding model.


In [ ]:
ctx_sentence = ServiceContext.from_defaults(llm=llm, embed_model=embed_model, node_parser=sentence_node_parser)

In [ ]:
ctx_base = ServiceContext.from_defaults(llm=llm, embed_model=embed_model, node_parser=base_node_parser)

## Section 7. Storing embeddings in a Qdrant vector store

So far the nodes only exist in memory. To actually perform similarity search efficiently, their embeddings need to be stored in a vector database. This notebook uses Qdrant, a vector database that supports fast nearest neighbor search over embedding vectors, along with any metadata attached to them.

Two separate collections are created in the same Qdrant instance, one to hold the sentence window nodes and one to hold the base nodes, so the two approaches remain cleanly separated even though they share the same database.


### Connecting to Qdrant

The client below points at a hosted Qdrant Cloud instance and authenticates with an API key. As with the Google API key earlier, this key should be loaded from an environment variable or a secrets manager rather than hard coded, since committing a live database key into a notebook or a GitHub repository would expose it to anyone who can view the file.


In [ ]:
from llama_index.vector_stores.qdrant import QdrantVectorStore

In [ ]:
import qdrant_client

In [ ]:
import os

client = qdrant_client.QdrantClient(
    url=os.environ.get("QDRANT_URL", "YOUR_QDRANT_CLOUD_URL_HERE"),
    api_key=os.environ.get("QDRANT_API_KEY", "YOUR_QDRANT_API_KEY_HERE"),  # For Qdrant Cloud, None for local instance
)

An alternative, shown for reference below (and commented out), is to run Qdrant fully in memory for quick, lightweight experiments that do not require anything to be deployed. This requires `qdrant client` version 1.1.1 or later, and does not persist data once the process ends.


In [ ]:
"""
client = qdrant_client.QdrantClient(
    # you can use :memory: mode for fast and lightweight experiments,
    # it does not require Qdrant to be deployed anywhere,
    # but it does require qdrant client >= 1.1.1
    # location=":memory:"
    # otherwise, set the Qdrant instance address with:
    # url="http://localhost:6333"
)
"""


### Creating the sentence window collection and index

`QdrantVectorStore` wraps the client above and points it at a named collection, here called `got_sent_node`. A `StorageContext` then ties that vector store to the index building process, and `VectorStoreIndex.from_documents` performs the full pipeline in one call. It parses the documents using the node parser attached to `ctx_sentence` (the sentence window parser), embeds each resulting node with the configured embedding model, and stores the resulting vectors in the Qdrant collection.


In [ ]:
vector_store = QdrantVectorStore(client=client, collection_name="got_sent_node")

In [ ]:
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context, service_context=ctx_sentence)

### Creating the base collection and index

The same sequence of steps is repeated for the base chunking strategy, using a second Qdrant collection called `got_base_node` and the `ctx_base` service context, which is configured with the plain `SentenceSplitter` instead of the sentence window parser.


In [ ]:
vector_store2 = QdrantVectorStore(client=client, collection_name="got_base_node")

In [ ]:
storage_context2 = StorageContext.from_defaults(vector_store=vector_store2)

In [ ]:
index2 = VectorStoreIndex.from_documents(documents, storage_context=storage_context2, service_context=ctx_base)

### Persisting and reloading an index

Once an index has been built, its underlying storage context can be persisted to disk so it does not need to be rebuilt from scratch in a future session. The cells below (kept as reference, and left inactive) show the typical pattern for saving an index to a local folder and loading it back afterward using `StorageContext.from_defaults(persist_dir=...)` and `load_index_from_storage`.


In [ ]:
"""
sentence_index.storage_context.persist(persist_dir="./sentence_index")
base_index.storage_context.persist(persist_dir="./base_index")
"""


In [ ]:
"""
# Download the persisted index folders to your own computer as a backup
!zip -r ./indexes.zip ./*_index

from google.colab import files
files.download("./indexes.zip")
"""


In [ ]:
"""
# rebuild the storage context from a previously persisted directory
SC_retrieved_sentence = StorageContext.from_defaults(persist_dir="./sentence_index")
SC_retrieved_base = StorageContext.from_defaults(persist_dir="./base_index")
"""


In [ ]:
"""
# load the index back from the rebuilt storage context
retrieved_sentence_index = load_index_from_storage(SC_retrieved_sentence)
retrieved_base_index = load_index_from_storage(SC_retrieved_base)
"""


## Section 8. Building the query engines

A query engine is the object that actually answers a natural language question. It takes the incoming question, embeds it, retrieves the closest matching nodes from the vector store, optionally runs those nodes through one or more postprocessors, and finally passes the resulting context to the language model to produce a final answer.

Two query engines are built here, one per index, and both retrieve the top 3 most similar nodes (`similarity_top_k=3`).


### The sentence window query engine

This is where the second half of the sentence window technique comes into play. The `node_postprocessors` argument attaches a `MetadataReplacementPostProcessor`, configured with `target_metadata_key="window"`. After the initial similarity search retrieves the best matching single sentences, this postprocessor swaps each node's text out for the full window of surrounding sentences stored under that metadata key, before anything is sent to the language model. This is the step that gives the language model the wider context it needs, even though the search itself was based on narrow, single sentence embeddings.


In [ ]:
sentence_query_engine = index.as_query_engine(
    similarity_top_k=3,
    verbose=True,
    # the target key defaults to window, matching the node parser default
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)

### The base query engine

The base query engine has no postprocessor at all. It simply retrieves the top 3 chunks that the plain `SentenceSplitter` produced and passes them directly to the language model as they were originally chunked.


In [ ]:
base_query_engine = index2.as_query_engine(
    similarity_top_k=3,
    verbose=True
)

## Section 9. Asking questions and comparing the answers

With both query engines ready, the same question can now be sent to each one so their answers can be compared directly. Any difference between the two answers can be attributed to the chunking and context strategy, since both engines share the same underlying language model, embedding model, and source document.


In [ ]:
question = "who is Ser Waymar Royce?"

### Answer from the base query engine

The base engine only has access to whatever larger chunks the plain sentence splitter happened to produce, so its answer reflects however much (or little) relevant detail landed inside those chunks.


In [ ]:
base_response = base_query_engine.query(
    question
)
print(base_response)

Example output produced by the base engine:

`Ser Waymar Royce is a young lord and a member of the Night's Watch. He is described as being tall and handsome, with long black hair and blue eyes. He is also said to be brave and skilled in combat.`


### Answer from the sentence window query engine

The sentence window engine searched using narrow, single sentence embeddings, but thanks to the metadata replacement postprocessor, the language model actually received the full surrounding windows of text for each retrieved sentence. This typically produces a noticeably richer, more detailed answer, since the model has access to more of the surrounding descriptive text.


In [ ]:
sentence_response = sentence_query_engine.query(
    question
)
print(sentence_response)

Example output produced by the sentence window engine:

`Ser Waymar Royce is the youngest son of an ancient house with too many heirs. He is a handsome youth of eighteen, grey eyed and graceful and slender as a knife. He is a knight who wears black leather boots, black woolen pants, black moleskin gloves, and a fine supple coat of gleaming black ringmail over layers of black wool and boiled leather. He is mounted on a huge black destrier and towers above Will and Gared on their smaller garrons.`


### Another example question

A second question is included here to show the same comparison can be repeated with any question relevant to the source document. Running this question through both engines follows exactly the same pattern as above.


In [ ]:
question = "How long have Gared and Will been part of the Night's Watch?"

## Conclusion

This notebook set up two parallel RAG pipelines over the same source document and the same underlying embedding and language models, differing only in how the document was split and how much context was handed to the language model after retrieval.

The base pipeline used a standard `SentenceSplitter`, embedding and retrieving whole multi sentence chunks. This is simple and works reasonably well in general, but it forces a tradeoff between chunk size and embedding precision, and it can miss surrounding detail if the useful information happens to fall just outside chunk boundaries.

The sentence window pipeline used `SentenceWindowNodeParser` to embed and search on single, precise sentences, and then used `MetadataReplacementPostProcessor` to substitute in a wider window of surrounding sentences right before the language model generates its answer. This combination kept the search step sharp while still giving the language model enough surrounding context to produce a fuller, more detailed answer, which was visible directly in the comparison of the two responses to the same question about Ser Waymar Royce.

The main takeaway is that the accuracy of the similarity search and the amount of context given to the language model do not have to be the same unit of text. Searching narrowly and then expanding context after retrieval, as sentence window retrieval does, is a practical way to get the benefits of both small and large chunks at once, and it is a useful technique to keep in mind whenever a RAG pipeline is producing answers that feel too shallow or too vague.

Some natural next steps for extending this notebook include experimenting with different window sizes to see how much surrounding context is actually useful, trying an automerging or hierarchical retriever as a further alternative strategy, adding a reranking step on top of either pipeline, and evaluating both approaches with a larger and more systematic set of questions rather than a single manual comparison.
